# Inference Baseline: Accuracy, mAP, Latency, Throughput & GPU Memory
**Runtime:** Google Colab → *Runtime ▸ Change runtime type ▸ T4 GPU*

A **baseline** is the "before" photo. Every later optimisation (FP16, TensorRT, batching, pruning) is judged against it. Without it, "faster" and "still accurate" are just opinions.

**Think of a bank branch:**
| Metric | Bank-counter meaning | Question it answers |
|---|---|---|
| **Accuracy / mAP** | How often the clerk gets the transaction *right* | Is the model correct? |
| **Latency** | How long *one* customer waits at the counter | How fast is a single request? |
| **Throughput** | Customers served *per hour* | How much work per second? |
| **GPU memory** | Size of the counter and desk space | Does it fit, and how many counters can I open? |

Latency and throughput are **not the same thing**. Opening more counters (bigger batches) raises customers-per-hour but can make each customer wait longer. The notebook measures that trade-off.

**What we measure**
1. Classification accuracy (top-1 / top-5): ResNet-50 on Imagenette
2. Detection mAP: YOLOv8 on COCO128
3. Latency: p50 / p95 / p99, cold start vs warm
4. Throughput: batch sweep and the "knee"
5. GPU memory: weights vs activations, peak, max batch
6. FP16 vs FP32 comparison
7. Save `baseline_report.json` + automatic regression check

In [ ]:
import torch, time, copy, json, os, tarfile, urllib.request, subprocess, platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
dev = torch.device("cuda")
torch.manual_seed(0); np.random.seed(0)
props = torch.cuda.get_device_properties(0)
ENV = dict(gpu=props.name, vram_gb=round(props.total_memory / 1e9, 1), torch=torch.__version__,
           cuda=torch.version.cuda, python=platform.python_version())
print(ENV)

def smi_mem_mb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().split(",")
    return int(out[0]), int(out[1])

print("nvidia-smi memory used/total (MiB):", smi_mem_mb())

## 1. Accuracy: classification (top-1 and top-5)
- **Top-1** = the model's single best guess is right.
- **Top-5** = the true class is among its 5 best guesses.

**Data:** *Imagenette*, 10 easy ImageNet classes. Because the model is an ImageNet-1000 classifier, we map each Imagenette folder to its ImageNet class index and count a prediction as correct only if the model picks *that exact index out of 1000*. No retraining, so this is a fair baseline.

In [ ]:
WNID_TO_IDX = {"n01440764": 0, "n02102040": 217, "n02979186": 482, "n03000684": 491, "n03028079": 497,
               "n03394916": 566, "n03417042": 569, "n03425413": 571, "n03445777": 574, "n03888257": 701}
NAMES = ["tench", "springer spaniel", "cassette player", "chain saw", "church",
         "French horn", "garbage truck", "gas pump", "golf ball", "parachute"]

if not os.path.isdir("imagenette2-160"):
    print("Downloading Imagenette (~100 MB)...")
    urllib.request.urlretrieve("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz", "imagenette.tgz")
    with tarfile.open("imagenette.tgz") as t: t.extractall(".")

tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
ds = datasets.ImageFolder("imagenette2-160/val", transform=tf)
cls_idx = torch.tensor([WNID_TO_IDX[c] for c in ds.classes])      # folder index -> ImageNet index
N = min(2000, len(ds))
idx = np.random.RandomState(0).permutation(len(ds))[:N].tolist()  # fixed subset = reproducible baseline
loader = DataLoader(Subset(ds, idx), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print("Evaluating on", N, "images")

resnet = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1).eval().to(dev)

@torch.inference_mode()
def eval_classifier(model, fp16=False):
    hit1 = hit5 = n = 0
    pc_hit, pc_n = np.zeros(10), np.zeros(10)
    for x, y in loader:
        x = x.to(dev, non_blocking=True)
        if fp16: x = x.half()
        probs = model(x).float().softmax(1)
        target = cls_idx[y].to(dev)
        top5 = probs.topk(5, 1).indices
        h1 = (top5[:, 0] == target)
        h5 = (top5 == target[:, None]).any(1)
        hit1 += h1.sum().item(); hit5 += h5.sum().item(); n += len(y)
        yc, hc = y.numpy(), h1.cpu().numpy()
        for c in range(10):
            m = yc == c
            pc_hit[c] += hc[m].sum(); pc_n[c] += m.sum()
    return dict(top1=100 * hit1 / n, top5=100 * hit5 / n, n=n), pc_hit / np.maximum(pc_n, 1) * 100

acc32, per_class32 = eval_classifier(resnet)
print(f"FP32  top-1: {acc32['top1']:.2f}%   top-5: {acc32['top5']:.2f}%   (n={acc32['n']})")

plt.figure(figsize=(9, 3.5))
plt.bar(NAMES, per_class32, color="steelblue"); plt.xticks(rotation=35, ha="right")
plt.ylabel("top-1 %"); plt.title("Per-class accuracy (find the weak classes)"); plt.ylim(0, 105); plt.tight_layout(); plt.show()

## 2. Accuracy: detection (mAP)
Detection must get **where** (box) and **what** (class) right.

- **IoU** = overlap of predicted box and true box ÷ their union. A hit needs IoU above a threshold.
- **Precision** = of the boxes I drew, how many were right. **Recall** = of the real objects, how many I found.
- **AP** = area under the precision-recall curve for one class. **mAP** = mean of AP over all classes.
- **mAP@50** uses IoU ≥ 0.5 (lenient). **mAP@50-95** averages IoU from 0.5 to 0.95 in steps of 0.05 (strict, the headline COCO number).

> **Caveat:** COCO128 is 128 images taken from COCO *train*, which YOLOv8 has already seen, so the score is a quick sanity baseline and is optimistic. For an official number set `FULL_COCO = True` (downloads ~1 GB, takes longer).

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO

FULL_COCO = False
DATA = "coco.yaml" if FULL_COCO else "coco128.yaml"
yolo = YOLO("yolov8n.pt")

def run_val(half):
    m = yolo.val(data=DATA, imgsz=640, batch=16, device=0, half=half, workers=2, plots=False, verbose=False)
    return dict(map50_95=float(m.box.map), map50=float(m.box.map50), map75=float(m.box.map75),
                precision=float(m.box.mp), recall=float(m.box.mr), speed_ms=dict(m.speed)), m

det32, m32 = run_val(half=False)
print({k: round(v, 4) for k, v in det32.items() if k != "speed_ms"})
print("Ultralytics stage timing (ms/img):", {k: round(v, 2) for k, v in det32["speed_ms"].items()})

In [ ]:
# Per-class AP: which classes does the detector struggle with?
try:
    ci = list(m32.box.ap_class_index)
    ap = np.array(m32.box.maps)[ci]
    names = [m32.names[i] for i in ci]
    d = pd.Series(ap, index=names).sort_values()
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    d.head(10).plot(kind="barh", ax=ax[0], color="indianred"); ax[0].set_title("10 weakest classes (mAP50-95)")
    d.tail(10).plot(kind="barh", ax=ax[1], color="seagreen"); ax[1].set_title("10 strongest classes (mAP50-95)")
    plt.tight_layout(); plt.show()
except Exception as ex:
    print("Per-class plot skipped:", ex)

## 3. Latency
**Latency = time for ONE request.** Averages hide problems, so we report **percentiles**:
- **p50** = typical request. **p95 / p99** = the slow tail users actually notice.
- **Cold start** = the very first call (CUDA context, cuDNN kernel selection). Never mix it into the steady-state number.

We time with **CUDA events** (GPU-side clock) and also with a **wall clock** (what a caller sees, including launch overhead).

In [ ]:
ymodel = YOLO("yolov8n.pt").model.float().to(dev).eval()
resnet32 = copy.deepcopy(resnet).float().eval()

@torch.inference_mode()
def latency_samples(model, x, iters=200, warmup=30):
    for _ in range(warmup): model(x)
    torch.cuda.synchronize()
    gpu, wall = [], []
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    for _ in range(iters):
        t0 = time.perf_counter(); s.record(); model(x); e.record(); e.synchronize()
        wall.append((time.perf_counter() - t0) * 1e3); gpu.append(s.elapsed_time(e))
    return np.array(gpu), np.array(wall)

def stats(a):
    p = np.percentile(a, [50, 90, 95, 99])
    return dict(mean=a.mean(), std=a.std(), min=a.min(), p50=p[0], p90=p[1], p95=p[2], p99=p[3], max=a.max())

@torch.inference_mode()
def cold_start(make_model, x):
    m = make_model()
    torch.cuda.synchronize(); t = time.perf_counter(); m(x); torch.cuda.synchronize()
    first = (time.perf_counter() - t) * 1e3
    t = time.perf_counter(); m(x); torch.cuda.synchronize()
    return first, (time.perf_counter() - t) * 1e3

x_cls = torch.randn(1, 3, 224, 224, device=dev)
x_det = torch.randn(1, 3, 640, 640, device=dev)

lat = {}
for name, model, x in [("resnet50 @224", resnet32, x_cls), ("yolov8n @640", ymodel, x_det)]:
    g, w = latency_samples(model, x)
    lat[name] = dict(gpu=g, wall=w)

tbl = pd.DataFrame({f"{n} [{k}]": stats(v[k]) for n, v in lat.items() for k in ("gpu", "wall")}).T
display(tbl.round(3))

first, second = cold_start(lambda: copy.deepcopy(resnet32).to(dev).eval(), x_cls)
print(f"Cold start (1st call): {first:.1f} ms | 2nd call: {second:.1f} ms | cold penalty: {first/second:.0f}x")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 7))
for r, (name, d) in enumerate(lat.items()):
    ax[r, 0].hist(d["gpu"], bins=40, color="steelblue")
    for q, c in [(50, "green"), (95, "orange"), (99, "red")]:
        ax[r, 0].axvline(np.percentile(d["gpu"], q), color=c, ls="--", label=f"p{q}")
    ax[r, 0].set_title(f"{name}: latency distribution (GPU ms)"); ax[r, 0].legend()
    ax[r, 1].plot(d["gpu"], lw=0.8); ax[r, 1].set_title(f"{name}: per-request latency (spikes = jitter)")
    ax[r, 1].set_xlabel("request #"); ax[r, 1].set_ylabel("ms")
plt.tight_layout(); plt.show()
print("Gap between 'wall' and 'gpu' = Python + kernel-launch + sync overhead.")

## 4. Throughput
**Throughput = images per second.** Batching lets the GPU do many images per kernel launch, so throughput rises, but each *batch* takes longer, so per-request latency rises too.

The **knee** = the smallest batch that reaches ≥ 90 % of peak throughput. Beyond it you pay more latency and memory for almost no gain.

The sweep records **memory at the same time** (used in Section 5).

In [ ]:
@torch.inference_mode()
def sweep(model, make_input, batches, iters=20, warmup=5):
    rows = []
    for B in batches:
        try:
            torch.cuda.empty_cache(); torch.cuda.synchronize()
            base = torch.cuda.memory_allocated()               # weights already resident
            torch.cuda.reset_peak_memory_stats()
            x = make_input(B)
            for _ in range(warmup): model(x)
            torch.cuda.synchronize()
            s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            s.record()
            for _ in range(iters): model(x)
            e.record(); e.synchronize()
            ms = s.elapsed_time(e) / iters
            peak = torch.cuda.max_memory_allocated()
            rows.append(dict(B=B, batch_ms=ms, ms_per_img=ms / B, img_per_s=B / ms * 1e3,
                             base_MB=base / 2**20, peak_MB=peak / 2**20, act_MB=(peak - base) / 2**20,
                             reserved_MB=torch.cuda.memory_reserved() / 2**20))
            del x
        except torch.cuda.OutOfMemoryError:
            print(f"OOM at batch {B}"); torch.cuda.empty_cache(); break
    return pd.DataFrame(rows).set_index("B")

def knee(df):
    return int(df.index[df.img_per_s >= 0.9 * df.img_per_s.max()][0])

mk_cls = lambda B: torch.randn(B, 3, 224, 224, device=dev)
mk_det = lambda B: torch.randn(B, 3, 640, 640, device=dev)
sw_cls = sweep(resnet32, mk_cls, [1, 2, 4, 8, 16, 32, 64, 128, 256])
sw_det = sweep(ymodel, mk_det, [1, 2, 4, 8, 16, 32, 64])

for name, df in [("ResNet-50 @224", sw_cls), ("YOLOv8n @640", sw_det)]:
    print(f"\n=== {name} === peak throughput {df.img_per_s.max():.0f} img/s | knee at batch {knee(df)}")
    display(df[["batch_ms", "ms_per_img", "img_per_s"]].round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for a, (name, df) in zip(ax, [("ResNet-50 @224", sw_cls), ("YOLOv8n @640", sw_det)]):
    a.plot(df.index, df.img_per_s, "o-", color="tab:blue"); a.set_xscale("log", base=2)
    a.set_xlabel("batch size"); a.set_ylabel("throughput (img/s)", color="tab:blue"); a.grid(True, alpha=.3)
    a2 = a.twinx(); a2.plot(df.index, df.batch_ms, "s--", color="tab:red"); a2.set_ylabel("batch latency (ms)", color="tab:red")
    a.axvline(knee(df), color="green", ls=":", label=f"knee = {knee(df)}"); a.legend(loc="center left")
    a.set_title(name + ": throughput up, latency up")
plt.tight_layout(); plt.show()

## 5. GPU memory
Where does GPU memory go during inference?
- **Weights**: fixed cost (parameters × bytes per parameter).
- **Activations**: intermediate tensors; grows **linearly with batch size**.
- **CUDA context + allocator cache**: overhead that PyTorch's counters do not show but `nvidia-smi` does.

Three different numbers, three different meanings:
| Counter | Meaning |
|---|---|
| `memory_allocated` | bytes in live tensors right now |
| `max_memory_allocated` | **peak** live bytes (what you need to *fit*) |
| `memory_reserved` | what PyTorch's caching allocator holds from the driver |

In [ ]:
def weight_mb(m): return sum(p.numel() * p.element_size() for p in m.parameters()) / 2**20

for name, model, df in [("ResNet-50", resnet32, sw_cls), ("YOLOv8n", ymodel, sw_det)]:
    slope, icpt = np.polyfit(df.index.values, df.act_MB.values, 1)
    print(f"{name}: weights = {weight_mb(model):.0f} MB | activations ≈ {slope:.1f} MB per extra image (linear fit)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, (name, df) in zip(ax, [("ResNet-50 @224", sw_cls), ("YOLOv8n @640", sw_det)]):
    a.stackplot(df.index, df.base_MB, df.act_MB, labels=["weights", "activations (peak - weights)"], colors=["#4C72B0", "#DD8452"])
    a.plot(df.index, df.reserved_MB, "k--", label="reserved by allocator"); a.set_xscale("log", base=2)
    a.set_xlabel("batch size"); a.set_ylabel("MB"); a.set_title(name + ": GPU memory"); a.legend()
plt.tight_layout(); plt.show()

used, total = smi_mem_mb()
print(f"\nnvidia-smi: {used} / {total} MiB used | PyTorch reserved: {torch.cuda.memory_reserved()/2**20:.0f} MiB | "
      f"allocated: {torch.cuda.memory_allocated()/2**20:.0f} MiB")
print("Gap between nvidia-smi and PyTorch = CUDA context (+ other models loaded in this session).")

In [ ]:
# How big a batch fits? Doubling probe (capped so it stays quick). Peak memory is what matters.
@torch.inference_mode()
def max_batch(model, make_input, cap=2048):
    B, last_ok = 1, 0
    while B <= cap:
        try:
            torch.cuda.empty_cache(); x = make_input(B); model(x); torch.cuda.synchronize(); last_ok = B; del x; B *= 2
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); break
    return last_ok

mb_cls = max_batch(resnet32, mk_cls)
print("Largest power-of-two batch that fits (ResNet-50 FP32, cap 2048):", mb_cls)

## 6. FP16 vs FP32: the trade-off in one table
Half precision halves weight/activation bytes and uses tensor cores. The cost is a possible small accuracy change. A baseline lets us *prove* the change is acceptable.

In [ ]:
resnet16 = copy.deepcopy(resnet).half().eval()
acc16, _ = eval_classifier(resnet16, fp16=True)
det16, _ = run_val(half=True)

g16, w16 = latency_samples(resnet16, x_cls.half())
sw16 = sweep(resnet16, lambda B: torch.randn(B, 3, 224, 224, device=dev).half(), [1, 2, 4, 8, 16, 32, 64, 128, 256])

def row(tag, acc, det, g, sw):
    return {"precision": tag, "top1_%": acc["top1"], "top5_%": acc["top5"], "mAP50-95": det["map50_95"],
            "lat_p50_ms (B=1)": np.percentile(g, 50), "lat_p95_ms (B=1)": np.percentile(g, 95),
            "peak_img/s": sw.img_per_s.max(), "peak_MB @B=64": sw.loc[64, "peak_MB"], "weights_MB": None}

cmp = pd.DataFrame([row("FP32", acc32, det32, lat["resnet50 @224"]["gpu"], sw_cls),
                    row("FP16", acc16, det16, g16, sw16)]).set_index("precision")
cmp["weights_MB"] = [weight_mb(resnet32), weight_mb(resnet16)]
display(cmp.round(3))
print(f"Speed-up (throughput): {cmp.loc['FP16','peak_img/s']/cmp.loc['FP32','peak_img/s']:.2f}x | "
      f"Memory @B=64: {cmp.loc['FP16','peak_MB @B=64']/cmp.loc['FP32','peak_MB @B=64']:.2f}x | "
      f"Top-1 change: {cmp.loc['FP16','top1_%']-cmp.loc['FP32','top1_%']:+.2f} pt")

## 7. Save the baseline and build a regression gate
All FP32 numbers go into `baseline_report.json`. Any future change is checked against it automatically, so you never ship a "faster" model that quietly lost accuracy.

In [ ]:
def flat(acc, det, gpu_lat, sw, peak_b=64):
    return {"top1": acc["top1"], "map50_95": det["map50_95"], "p50_ms": float(np.percentile(gpu_lat, 50)),
            "p95_ms": float(np.percentile(gpu_lat, 95)), "peak_img_per_s": float(sw.img_per_s.max()),
            "peak_mem_MB": float(sw.loc[peak_b, "peak_MB"])}

baseline = dict(
    env=ENV,
    classifier=dict(model="resnet50 IMAGENET1K_V1", dataset="Imagenette val subset", **acc32),
    detector=dict(model="yolov8n", dataset=DATA, **{k: v for k, v in det32.items()}),
    latency={n: dict(gpu=stats(d["gpu"]), wall=stats(d["wall"])) for n, d in lat.items()},
    cold_start_ms=dict(first=first, second=second),
    throughput=dict(resnet50=sw_cls[["batch_ms", "img_per_s"]].to_dict("index"),
                    yolov8n=sw_det[["batch_ms", "img_per_s"]].to_dict("index"),
                    resnet50_knee_batch=knee(sw_cls), yolov8n_knee_batch=knee(sw_det)),
    memory=dict(resnet50_weights_MB=weight_mb(resnet32), yolov8n_weights_MB=weight_mb(ymodel),
                resnet50_peak_MB=sw_cls.peak_MB.to_dict(), yolov8n_peak_MB=sw_det.peak_MB.to_dict(),
                resnet50_max_pow2_batch=mb_cls),
    gate_metrics=flat(acc32, det32, lat["resnet50 @224"]["gpu"], sw_cls),
)
conv = lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o)
with open("baseline_report.json", "w") as f: json.dump(baseline, f, indent=2, default=conv)
print("Saved baseline_report.json")

# ---- regression gate ----
RULES = {"top1": ("higher", -0.5, "pt"), "map50_95": ("higher", -0.005, "abs"),
         "p50_ms": ("lower", 10, "%"), "p95_ms": ("lower", 10, "%"),
         "peak_img_per_s": ("higher", -10, "%"), "peak_mem_MB": ("lower", 10, "%")}

def regression_check(new, base):
    rows = []
    for k, (direction, tol, unit) in RULES.items():
        b, n = base[k], new[k]
        delta = (n - b) if unit in ("pt", "abs") else (n - b) / b * 100
        ok = delta >= tol if direction == "higher" else delta <= tol
        rows.append(dict(metric=k, baseline=round(b, 4), new=round(n, 4), change=f"{delta:+.3f} {unit}",
                         allowed=f"{tol:+} {unit}", result="PASS" if ok else "FAIL"))
    return pd.DataFrame(rows).set_index("metric")

fp16_metrics = flat(acc16, det16, g16, sw16)
print("Candidate: FP16 ResNet-50 / YOLOv8n  vs  FP32 baseline")
display(regression_check(fp16_metrics, baseline["gate_metrics"]))

## 8. Report summary (fill in for your write-up)
| Metric | Value (from this run) | Where |
|---|---|---|
| GPU / CUDA / PyTorch | `ENV` | Section 0 |
| Top-1 / Top-5 (ResNet-50) | `acc32` | Section 1 |
| mAP50 / mAP50-95 (YOLOv8n) | `det32` | Section 2 |
| Latency p50 / p95 / p99 (B=1) | `tbl` | Section 3 |
| Cold-start penalty | `first / second` | Section 3 |
| Peak throughput and knee batch | `sw_cls`, `knee()` | Section 4 |
| Weights MB, MB per image, max batch | memory cells | Section 5 |
| FP16 gain and accuracy cost | `cmp` | Section 6 |
| Regression verdict | `regression_check` | Section 7 |

**Key takeaways to remember**
1. **Report percentiles, not averages.** p95/p99 describe the slow tail.
2. **Warm up, then measure**, and always report cold start separately.
3. **Latency and throughput trade off** through batch size; find the knee.
4. **Memory = weights (fixed) + activations (grows with batch).**
5. **Optimise only against a baseline**, and re-check accuracy every time.